In [0]:


%python
from datetime import datetime, timezone

dbutils.widgets.text("catalog","dev","Target catalog")

dbutils.widgets.text("run_date",datetime.now(timezone.utc).date().isoformat(),"Run date (YYYY-MM-DD)")

catalog = dbutils.widgets.get("catalog")
run_date = dbutils.widgets.get("run_date")


In [0]:
%python
table_prefix = f"{catalog}.stepright"

BRONZE_TABLES = [
    f"{table_prefix}.bronze_orders",
    f"{table_prefix}.bronze_order_items",
    f"{table_prefix}.bronze_customers",
    f"{table_prefix}.bronze_products",
    f"{table_prefix}.bronze_categories",
    f"{table_prefix}.bronze_inventory",
    f"{table_prefix}.bronze_clickstream",
]

QUARANTINE_PAIRS = [
    (f"{table_prefix}.bronze_orders", f"{table_prefix}.bronze_orders_quarantined"),
    (f"{table_prefix}.bronze_order_items", f"{table_prefix}.bronze_order_items_quarantined"),
    (f"{table_prefix}.bronze_customers", f"{table_prefix}.bronze_customers_quarantined"),
    (f"{table_prefix}.bronze_products", f"{table_prefix}.bronze_products_quarantined"),
    (f"{table_prefix}.bronze_categories", f"{table_prefix}.bronze_categories_quarantined"),
    (f"{table_prefix}.bronze_inventory", f"{table_prefix}.bronze_inventory_quarantined"),
    (f"{table_prefix}.bronze_clickstream", f"{table_prefix}.bronze_clickstream_quarantined"),
]

QUARANTINE_RATE_THRESHOLD = 0.5

print(f"Loaded {len(BRONZE_TABLES)} bronze tables and {len(QUARANTINE_PAIRS)} quarantine pairs.")
print(f"Quarantine rate threshold: {QUARANTINE_RATE_THRESHOLD:.0%}")




def date_filter()-> str:
    return f"date(_ingested_at)='{run_date}'"

In [0]:
%python


failures=[]

for table in BRONZE_TABLES:
    count=spark.sql(f"select count(*) from {table} where {date_filter()}").collect()[0][0]
    if count==0:
        failures.append(table)
        print(f"Table {table} has no records for {run_date}")
        
if len(failures)>0:
    raise Exception(f"Found {len(failures)} tables with no records for {run_date}: {','.join(failures)}")


for valid_table,quarantined_table in QUARANTINE_PAIRS:
    filter_expr=date_filter()
    valid_count=spark.table(f"{catalog}.stepright.{valid_table}").filter(filter_expr).count()
    quarantined_count=spark.table(f"{catalog}.stepright.{quarantined_table}").filter(filter_expr).count()
    total=valid_count+quarantined_count
    if total>0:
        rate=quarantined_count/total
        if rate>QUARANTINE_RATE_THRESHOLD:
            failures.append(f"{quarantined_table} quarantine rate for run_date={run_date} is {rate:.0%}"
                            f"(threshold {QUARANTINE_RATE_THRESHOLD:.0%}) - {quarantined_count} of {total} rows. ")
            print(f"Table {quarantined_table} has {rate:.0%} quarantined records for {run_date}")
            
    



In [0]:
%python

if failures:
    message=f"DQ gate failed for run_date={run_date}:\n{',\n'.join(failures)}"
    print(message)
    raise Exception(message)
else:
    print("DQ gate passed for run_date={run_date }")
